# SVRDD · Weight decay deneyi

YOLOv5s'i aynı başlangıç ağırlıkları, veri bölünmesi, augmentasyon, batch ve seed ile yeniden eğitip tek bir ayarı karşılaştıracağız. Sonuç ölçütlerimiz, baseline'da ortalamanın altında kalan dört sınıfın ortalama AP50'si ve genel mAP50:95.

Colab'da önce **Çalışma zamanı → Çalışma zamanı türünü değiştir** menüsünden kullanmak istediğin GPU'yu seç. Ücretli bir hesabın varsa oturum açıkken kredi tüketilebilir. Hücreleri sırayla çalıştır. 80 epoch eğitim, **Eğitim** başlıklı hücreyle başlar.

Ekibin baseline'ı A100 40 GB ile 43,4 dakika sürdü. Başka GPU kullanırsan ilk iki epoch'tan itibaren tahmini kalan süreyi göreceksin. Tam karşılaştırma için iki eğitimi aynı GPU tipinde yap. A100 kullanırsan ekibin sakladığı baseline'dan yararlanabilirsin. T4 kullanırsan aynı notebook'la `0.0005` için de referans ölçümü alabilirsin.

## GPU, kod ve veri

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Eğitim için Colab'da bir GPU seçip bu hücreyi tekrar çalıştır.")
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}")
drive.mount("/content/drive")

work = Path("/content/svrdd_weight_decay")
work.mkdir(exist_ok=True)
commit = "af93934c67d861346520487c72c65f752ba94584"
repo = work / "ekip_kodu"
if not repo.exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/wudii456/415-SVRDD-Road-Damage-Detection-Project.git",
                    str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "-q", commit], check=True)
version = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert version == commit
yolo = repo / "yolov5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(yolo / "requirements.txt")], check=True)
os.environ["WANDB_MODE"] = "disabled"
print(f"Ekip kodu hazır: {version[:7]}")

Ekip Drive'ındaki YOLO arşivi okunabiliyorsa Colab'a kopyalanır. Gerekirse yaklaşık 2,9 GB'lık aynı arşiv Zenodo'dan Colab'ın geçici diskine indirilir. Arşiv, veri yayıncısının verdiği sağlama toplamıyla doğrulanır. [SVRDD, Zenodo](https://zenodo.org/records/10100129)

In [ ]:
import yaml

archive = work / "SVRDD_YOLO.zip"
shared = Path("/content/drive/MyDrive/svrdd/zenodo/SVRDD_YOLO.zip")
data_root = work / "SVRDD_YOLO"
ready = work / "veri_dogrulandi"
if not ready.exists():
    if not archive.exists():
        partial = archive.with_suffix(".part")
        if shared.is_file():
            shutil.copyfile(shared, partial)
        else:
            subprocess.run(["curl", "--fail", "--location", "--retry", "3", "--output", str(partial),
                            "https://zenodo.org/records/10100129/files/SVRDD_YOLO.zip?download=1"], check=True)
        partial.replace(archive)
    digest = hashlib.md5()
    with archive.open("rb") as src:
        for block in iter(lambda: src.read(8 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != "4b897aa7081e8d0fa07f3d330e86e936":
        archive.unlink()
        raise ValueError("Colab'daki arşiv doğrulanamadı. Kaynak ZIP'i kontrol edip veri hücresini yinele.")
    with zipfile.ZipFile(archive) as source:
        source.extractall(work)
    ready.touch()

names = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole",
         "manhole_cover", "longitudinal_patch", "transverse_patch"]
listed = set()
config = {"path": str(work), "nc": len(names), "names": names}
for split, expected in [("train", 6000), ("val", 1000), ("test", 1000)]:
    lines = (data_root / f"{split}.txt").read_text().splitlines()
    paths = [data_root / line.strip().replace("\\", "/") for line in lines if line.strip()]
    assert len(paths) == len(set(paths)) == expected and not listed.intersection(paths), split
    for path in paths:
        label = path.parent.parent / "labels" / (path.stem + ".txt")
        assert path.is_file() and label.is_file(), path
    listed.update(paths)
    manifest = work / f"{split}.txt"
    manifest.write_text("\n".join(map(str, paths)) + "\n")
    config[split] = str(manifest)
    print(f"{split}: {len(paths)}")
assert listed == set(data_root.glob("*/images/*.jpg"))
data_yaml = work / "svrdd.yaml"
data_yaml.write_text(yaml.safe_dump(config))

## Deney seçimi

İlk yeni değer `0.001`. Aynı GPU tipinde kendi baseline'ını almak için `0.0005`, taramayı tamamlamak için `0.005` seçilebilir. Her yeni eğitim için bu hücreyi tekrar çalıştır. Çıktılar kendi Drive'ındaki **SVRDD_weight_decay_deneylerim** altında yeni bir alt klasöre kaydedilir.

In [ ]:
WEIGHT_DECAY = 0.001
assert WEIGHT_DECAY in (0.0005, 0.001, 0.005)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
name = f"wd{WEIGHT_DECAY:g}_s0_{stamp}_{uuid.uuid4().hex[:6]}"
experiments = Path("/content/drive/MyDrive/SVRDD_weight_decay_deneylerim")
run = experiments / name
hyp = yaml.safe_load((yolo / "data/hyps/hyp.scratch-low.yaml").read_text())
assert hyp["weight_decay"] == 0.0005
hyp["weight_decay"] = WEIGHT_DECAY
hyp_path = work / f"{name}.yaml"
hyp_path.write_text(yaml.safe_dump(hyp))
print(f"{gpu} · weight decay {WEIGHT_DECAY} · 80 epoch · batch 32 · seed 0")
print(f"Sonuç konumu: {run}")

## Eğitim

Aşağıdaki hücre 80 epoch'luk işi başlatır. Başlatmadan önce seçtiğin GPU'yu ve Colab'ın gösterdiği kredi tüketimini kontrol et. İlk iki epoch tamamlandığında, gerçekleşen süreye göre kalan dakika tahmini yazılır.

In [ ]:
import csv

if run.exists():
    raise FileExistsError(f"Bu deney zaten başlatılmış: {run}. Yeni bir deney için seçim hücresini tekrar çalıştır.")
run.mkdir(parents=True)
(run / "deney.json").write_text(json.dumps({
    "weight_decay": WEIGHT_DECAY, "commit": commit, "gpu": gpu,
    "torch": torch.__version__, "cuda": torch.version.cuda, "seed": 0,
    "batch": 32, "epochs": 80, "img": 640, "weights": "yolov5s.pt",
}, indent=2))
cmd = [sys.executable, "-u", "train.py", "--img", "640", "--batch", "32", "--epochs", "80",
       "--data", str(data_yaml), "--weights", "yolov5s.pt", "--hyp", str(hyp_path),
       "--seed", "0", "--cache", "ram", "--workers", "8",
       "--name", name, "--project", str(experiments), "--exist-ok"]
log = run / "egitim.log"
result = run / "results.csv"
with log.open("w") as output, subprocess.Popen(
    cmd, cwd=yolo, stdout=output, stderr=subprocess.STDOUT
) as job:
    last_epoch = 0
    first_time = None
    try:
        while job.poll() is None:
            if result.exists():
                with result.open() as src:
                    rows = list(csv.reader(src))
                epochs = sum(len(row) == len(rows[0]) for row in rows[1:])
                now = time.monotonic()
                if epochs > last_epoch:
                    if first_time is None:
                        first_time, first_epoch = now, epochs
                    estimate = ""
                    if epochs > first_epoch:
                        left = (80 - epochs) * (now - first_time) / (epochs - first_epoch) / 60
                        estimate = f" · tahmini kalan {left:.0f} dakika"
                    print(f"{epochs}/80 epoch tamamlandı{estimate}", flush=True)
                    last_epoch = epochs
            time.sleep(5)
    except KeyboardInterrupt:
        job.terminate()
        raise
if job.returncode:
    print(log.read_text(errors="replace")[-4000:])
    raise RuntimeError(f"Eğitim durdu. Günlük: {log}")
with result.open() as src:
    assert len(list(csv.DictReader(src))) == 80
assert (run / "weights/best.pt").is_file()
print("80 epoch tamamlandı. Şimdi doğrulama hücresini çalıştır.")

## Doğrulama ve karşılaştırma

En yüksek doğrulama mAP50:95 ile seçilen `best.pt`, ekip notebook'undaki ayrı doğrulama hücresiyle aynı ayarlarda ölçülür. AP50 farkları yüzde puan cinsindendir. Ekranda ekip A100 referansıyla karşılaştırma görünür. Başka GPU kullandıysan aynı GPU'daki `0.0005` koşusunun `karsilastirma.csv` dosyasındaki **Bu deney** sütunuyla eşleştir. Tablodaki sayılar YOLO'nun doğrulama çıktısının yuvarlama hassasiyetindedir.

In [ ]:
import pandas as pd

results_csv = run / "results.csv"
if not results_csv.is_file() or len(pd.read_csv(results_csv)) != 80:
    raise RuntimeError("Karşılaştırma için önce 80 epoch tamamlanmalı.")

val_cmd = [sys.executable, "val.py", "--weights", str(run / "weights/best.pt"),
           "--data", str(data_yaml), "--img", "640", "--batch-size", "32", "--task", "val",
           "--project", str(run), "--name", "dogrulama", "--exist-ok"]
check = subprocess.run(val_cmd, cwd=yolo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(run / "dogrulama.log").write_text(check.stdout)
if check.returncode:
    print(check.stdout[-3500:])
    check.check_returncode()
scores = {}
for line in check.stdout.splitlines():
    fields = line.split()
    if len(fields) == 7 and fields[0] in [*names, "all"] and fields[1] == "1000":
        scores[fields[0]] = (float(fields[-2]), float(fields[-1]))
assert set(scores) == set(names) | {"all"}, "Sınıf sonuçlarını dogrulama.log içinde kontrol et."
baseline = dict(zip(names, [0.693, 0.639, 0.764, 0.659, 0.822, 0.750, 0.688]))
weak = [cls for cls in names if baseline[cls] < 0.716]
rows = [(f"{cls} AP50", baseline[cls], scores[cls][0]) for cls in names]
rows += [("Dört zayıf sınıf · ortalama AP50",
          sum(baseline[c] for c in weak) / len(weak),
          sum(scores[c][0] for c in weak) / len(weak)),
         ("Genel mAP50", 0.716, scores["all"][0]),
         ("Genel mAP50:95", 0.408, scores["all"][1])]
table = pd.DataFrame(rows, columns=["Ölçüt", "Ekip A100 baseline", "Bu deney"])
table["Fark (yüzde puan)"] = 100 * (table["Bu deney"] - table["Ekip A100 baseline"])
table.to_csv(run / "karsilastirma.csv", index=False)
print(f"Ölçümler kaydedildi: {run}/karsilastirma.csv")
display(table.round(4))

Sonuç dosyasını kontrol ettikten sonra **Çalışma zamanı → Çalışma zamanının bağlantısını kes ve sil** seçeneğiyle GPU oturumunu kapatabilirsin.